In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null


## Cognitive Science Rationale

The **Tower of London** tests planning and problem-solving (Shallice, 1982). Models must find the minimum number of moves to rearrange balls on pegs to match a goal state. This requires lookahead and means-ends analysis.


## Interpreting the Score

Score = move_efficiency (optimal/actual). 1.0 = always finds optimal solution.


### References
Shallice (1982), Miyake et al. (2000)


# Tower of London Planning Benchmark

Tests multi-step planning ability.
Model must find optimal move sequences to rearrange balls on pegs.

**Cognitive Science**: Shallice (1982), Owen et al. (1990)
**Key metrics**: Optimality ratio, depth scaling
**Human optimality**: ~85% at 3 moves, ~55% at 5 moves

In [ ]:
"""
Stimuli generator for Tower of London (ToL) planning benchmark.

Generates goal states at varying optimal move depths (3, 4, 5 moves).
Uses 3 pegs and 3 colored balls. Each peg has a capacity constraint:
- Peg A: holds 3 balls
- Peg B: holds 2 balls
- Peg C: holds 1 ball

This matches the classic Shallice (1982) setup.
"""

import random
from collections import deque
from itertools import permutations
from copy import deepcopy

# Pegs with capacity constraints
PEG_CAPACITY = {"A": 3, "B": 2, "C": 1}
BALLS = ["red", "blue", "green"]


def state_to_tuple(state):
    """Convert state dict to hashable tuple."""
    return tuple(tuple(state[p]) for p in ["A", "B", "C"])


def tuple_to_state(t):
    """Convert tuple back to state dict."""
    return {"A": list(t[0]), "B": list(t[1]), "C": list(t[2])}


def get_valid_moves(state):
    """Get all valid moves from current state."""
    moves = []
    pegs = ["A", "B", "C"]
    for src in pegs:
        if not state[src]:  # empty peg
            continue
        ball = state[src][-1]  # top ball
        for dst in pegs:
            if dst == src:
                continue
            if len(state[dst]) < PEG_CAPACITY[dst]:
                moves.append((src, dst, ball))
    return moves


def apply_move(state, move):
    """Apply a move and return new state."""
    src, dst, ball = move
    new_state = deepcopy(state)
    new_state[src].pop()
    new_state[dst].append(ball)
    return new_state


def bfs_optimal(start, goal):
    """Find optimal (shortest) move sequence from start to goal using BFS."""
    start_t = state_to_tuple(start)
    goal_t = state_to_tuple(goal)

    if start_t == goal_t:
        return []

    queue = deque([(start_t, [])])
    visited = {start_t}

    while queue:
        current_t, path = queue.popleft()
        current = tuple_to_state(current_t)

        for move in get_valid_moves(current):
            new_state = apply_move(current, move)
            new_t = state_to_tuple(new_state)

            new_path = path + [move]
            if new_t == goal_t:
                return new_path

            if new_t not in visited:
                visited.add(new_t)
                queue.append((new_t, new_path))

    return None  # unreachable


def generate_all_states():
    """Generate all valid states (3 balls distributed across 3 pegs respecting capacity)."""
    states = []
    pegs = ["A", "B", "C"]

    # Each ball can be on any peg (if capacity allows)
    # We place balls one at a time
    def place_balls(balls_remaining, current_state):
        if not balls_remaining:
            states.append(deepcopy(current_state))
            return

        ball = balls_remaining[0]
        for peg in pegs:
            if len(current_state[peg]) < PEG_CAPACITY[peg]:
                current_state[peg].append(ball)
                place_balls(balls_remaining[1:], current_state)
                current_state[peg].pop()

    place_balls(BALLS, {"A": [], "B": [], "C": []})
    return states


def generate_tol_problems(n_per_depth=5, seed=42):
    """
    Generate Tower of London problems at depths 3, 4, and 5.
    Returns problems grouped by optimal move count.
    """
    random.seed(seed)
    all_states = generate_all_states()

    # Find all pairs with known optimal depths
    problems_by_depth = {3: [], 4: [], 5: []}

    for start in all_states:
        for goal in all_states:
            if state_to_tuple(start) == state_to_tuple(goal):
                continue
            optimal = bfs_optimal(start, goal)
            if optimal and len(optimal) in problems_by_depth:
                problems_by_depth[len(optimal)].append({
                    "start": deepcopy(start),
                    "goal": deepcopy(goal),
                    "optimal_moves": len(optimal),
                    "optimal_solution": [(s, d, b) for s, d, b in optimal],
                })

    # Sample n_per_depth problems from each depth
    problems = []
    for depth in [3, 4, 5]:
        candidates = problems_by_depth[depth]
        random.shuffle(candidates)
        selected = candidates[:n_per_depth]
        for i, p in enumerate(selected):
            p["problem_id"] = f"tol_{depth}move_{i+1}"
        problems.extend(selected)

    return problems


def state_str(state):
    """Human-readable state description."""
    parts = []
    for peg in ["A", "B", "C"]:
        if state[peg]:
            balls = ", ".join(state[peg])
            parts.append(f"Peg {peg}: [{balls}] (bottom→top)")
        else:
            parts.append(f"Peg {peg}: [empty]")
    return "\n".join(parts)


TOL_PROBLEMS = generate_tol_problems(n_per_depth=5, seed=42)

if __name__ == "__main__":
    print(f"Generated {len(TOL_PROBLEMS)} Tower of London problems")
    for p in TOL_PROBLEMS:
        print(f"\n{p['problem_id']} ({p['optimal_moves']} moves):")
        print(f"  Start: {p['start']}")
        print(f"  Goal:  {p['goal']}")
        print(f"  Solution: {p['optimal_solution']}")


In [ ]:
"""
Executive Functions Benchmark 2: Tower of London (ToL) Planning

Tests planning ability — a core executive function component.

The model is given an initial arrangement of 3 colored balls on 3 pegs (with
capacity constraints) and a goal state. It must plan a sequence of moves to
reach the goal in the minimum number of moves.

Cognitive Science Basis:
- Tower of London (Shallice, 1982)
- Planning is a "look-ahead" executive process (Owen et al., 1990)
- Difficulty scales with optimal move depth (3 < 4 < 5 moves)
- Frontal patients show deficits at higher move depths (Shallice, 1982)

Metrics:
- Optimality ratio: mean(optimal_moves / actual_moves) per problem
- Validity rate: proportion of solutions with all legal moves reaching goal
- Depth scaling: does performance degrade at higher depths (as in humans)?

Score = 0.50 * optimality + 0.30 * validity + 0.20 * depth_scaling_bonus

Shortcut Resistance:
- Problems are procedurally generated, not from standard test batteries
- Capacity constraints prevent trivial solutions
- Multiple move depths test genuine planning vs. random search
"""

import kaggle_benchmarks as kbench
from dataclasses import dataclass, field
import numpy as np
import re
from copy import deepcopy
# Stimuli and helpers defined above


# ─── Structured Output Schema ──────────────────────────────────────

@dataclass
class ToLResponse:
    """Model's planned move sequence."""
    moves: list       # List of moves, each as "X→Y" (e.g., "A→B")
    reasoning: str    # Explanation of planning strategy


# ─── Move Validation ────────────────────────────────────────────────

def parse_moves(moves_raw) -> list:
    """Parse move list from model response into (src, dst) tuples."""
    parsed = []
    if isinstance(moves_raw, str):
        # Try to parse "A→B, B→C" or "A->B\nB->C" etc.
        moves_raw = re.findall(r'([ABC])\s*(?:→|->|to)\s*([ABC])', moves_raw, re.IGNORECASE)
        for src, dst in moves_raw:
            parsed.append((src.upper(), dst.upper()))
    elif isinstance(moves_raw, list):
        for m in moves_raw:
            if isinstance(m, str):
                match = re.search(r'([ABC])\s*(?:→|->|to)\s*([ABC])', m, re.IGNORECASE)
                if match:
                    parsed.append((match.group(1).upper(), match.group(2).upper()))
            elif isinstance(m, (list, tuple)) and len(m) >= 2:
                parsed.append((str(m[0]).upper(), str(m[1]).upper()))
    return parsed


def validate_solution(start_state, goal_state, moves) -> dict:
    """
    Validate a sequence of moves.
    Returns dict with: valid (bool), reached_goal (bool), n_moves, errors list.
    """
    state = deepcopy(start_state)
    errors = []

    for i, (src, dst) in enumerate(moves):
        # Check source peg has balls
        if not state.get(src) or len(state[src]) == 0:
            errors.append(f"Move {i+1}: Peg {src} is empty")
            continue

        # Check destination has capacity
        if len(state.get(dst, [])) >= PEG_CAPACITY.get(dst, 0):
            errors.append(f"Move {i+1}: Peg {dst} is full (capacity {PEG_CAPACITY[dst]})")
            continue

        # Apply move
        ball = state[src].pop()
        state[dst].append(ball)

    reached_goal = state_to_tuple(state) == state_to_tuple(goal_state)

    return {
        "valid": len(errors) == 0,
        "reached_goal": reached_goal and len(errors) == 0,
        "n_moves": len(moves),
        "errors": errors,
        "final_state": state,
    }


# ─── The Benchmark Task ────────────────────────────────────────────

@kbench.task(name="exec_func_tol")
def exec_func_tol(llm) -> float:
    """
    Tower of London Planning Benchmark.

    Tests multi-step planning by requiring the model to find move sequences
    to rearrange balls on pegs to match a goal state.

    Score = 0.50 * optimality + 0.30 * validity + 0.20 * depth_scaling_bonus

    Cognitive Science Basis: Shallice (1982), Owen et al. (1990).
    Human optimality: ~85% at 3 moves, ~65% at 5 moves.
    """
    results = []
    depth_scores = {3: [], 4: [], 5: []}

    for problem in TOL_PROBLEMS:
        start = problem["start"]
        goal = problem["goal"]
        optimal = problem["optimal_moves"]

        prompt = (
            f"TOWER OF LONDON PUZZLE — {problem['problem_id']}\n\n"
            f"Rules:\n"
            f"- 3 pegs (A, B, C) with capacity limits: A holds 3 balls, B holds 2, C holds 1\n"
            f"- Move only the TOP ball from one peg to another\n"
            f"- Goal: reach the goal state in as FEW moves as possible\n"
            f"- Optimal solution needs {optimal} moves\n\n"
            f"CURRENT STATE:\n{state_str(start)}\n\n"
            f"GOAL STATE:\n{state_str(goal)}\n\n"
            f"Plan your moves carefully. List each move as 'X→Y' (e.g., 'A→B' means "
            f"move top ball from peg A to peg B).\n\n"
            f"Provide your moves as a list and explain your reasoning."
        )

        with kbench.chats.new(f"tol_{problem['problem_id']}"):
            try:
                response = llm(prompt, response_format=ToLResponse)
                moves_raw = response.moves
                reasoning = response.reasoning
            except Exception:
                raw = llm(prompt)
                moves_raw = raw
                reasoning = raw

        # Parse and validate
        moves = parse_moves(moves_raw)
        validation = validate_solution(start, goal, moves)

        # Compute optimality ratio (capped at 1.0)
        if validation["reached_goal"]:
            optimality = min(1.0, optimal / max(validation["n_moves"], 1))
        else:
            optimality = 0.0

        result = {
            "problem_id": problem["problem_id"],
            "optimal_moves": optimal,
            "model_moves": validation["n_moves"],
            "valid_moves": validation["valid"],
            "reached_goal": validation["reached_goal"],
            "optimality": round(optimality, 4),
            "errors": validation["errors"],
        }
        results.append(result)
        depth_scores[optimal].append(optimality)

    # ── Compute Metrics ──

    # Overall validity rate
    validity = sum(1 for r in results if r["reached_goal"]) / len(results)

    # Overall optimality (only counting valid solutions, but 0 for invalid)
    optimality_scores = [r["optimality"] for r in results]
    mean_optimality = np.mean(optimality_scores)

    # Depth scaling bonus: do scores decrease with depth? (as expected in humans)
    depth_means = {d: np.mean(s) if s else 0 for d, s in depth_scores.items()}
    # Bonus if 3-move > 4-move > 5-move (expected pattern)
    if depth_means[3] > depth_means[4] > depth_means[5]:
        depth_bonus = 1.0  # Shows human-like scaling
    elif depth_means[3] > depth_means[5]:
        depth_bonus = 0.5  # Partial scaling
    else:
        depth_bonus = 0.0  # No scaling or inverse

    # ── Composite Score ──
    score = (
        0.50 * float(mean_optimality) +
        0.30 * float(validity) +
        0.20 * float(depth_bonus)
    )
    score = round(float(np.clip(score, 0, 1)), 4)

    # ── Log ──
    kbench.log({
        "benchmark": "Tower of London",
        "n_problems": len(results),
        "overall_validity": round(float(validity), 4),
        "mean_optimality": round(float(mean_optimality), 4),
        "depth_scaling": {str(d): round(float(m), 4) for d, m in depth_means.items()},
        "depth_bonus": depth_bonus,
        "composite_score": score,
        "per_problem": results,
    })

    return score

exec_func_tol.run(llm=kbench.llm)
